In [ ]:
# importing the required packages
import czifile # to import a .czi file
# from PIL import Image # to convert .czi file to a .tif file
import skimage # general package for manipulating imaging data
from pathlib import Path # for file path 
import numpy as np
import matplotlib.pyplot as plt
from microfilm.microplot import microshow # for viewing multichannel image data
from skimage.transform import rotate # to rotate the image as a control
from skimage.restoration import rolling_ball # for image processing
from skimage.filters import gaussian # for image processing
from skimage.feature import peak_local_max # for local max detection
import sys

In [ ]:
# with this peice of code, it will recognize the custom modules
project_root = "/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting"
sys.path.append(project_root)

# custom modules
from synapse_counting import metadata, preprocessing, calc_synaptic_metrics

In [ ]:
file = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA1_SLM.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre, post = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)

p = preprocessing.ImagePreprocessing(
            include_rolling_ball = True, radius = 10, # rolling ball parameters
            include_clahe = False, clip_limit = 0.005, kernel_size = 150, nbins = 265, # CLAHE parameters
            include_tophat = True, element_size = 5, # tophat parameters
            include_blur = True, sigma = 1, preserve_range = True # gaussian blur filters
            )
pre_1, post_1 = p.preprocess(pre, post)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(pre_1, ax=axs[0], label_text = 'VLGUT1')
microshow(post_1, ax=axs[1], label_text = 'PSD95')

In [ ]:
def local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold):
    
    """Detects the local intensity peak of each channel processed
    
        Args:
            vglut1_preprocessed (np.array): processed image of vlgut1, which is background substracted and has a gaussian blur
            psd95_preprocessed (np.array): processed image of psd95, which is background substracted and has a gaussian blur
            vglut1_threshold (float): thresholding of the vlgut1 image for local peak detection
            psd95_threshold (float): thresholding of the psd95 image for local peak detection
    
        Returns:
            vglut1_coord (np.array): coordinates of vglut1 local peak maxima
            psd95_coord (np.array): coordinates of psd95 local peak maxima
            psd95_rot_coord (np.array): coordinates of psd95 local peak maxima
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    vglut1_coord = peak_local_max(vglut1_preprocessed, min_distance = 1, threshold_abs=vglut1_threshold)
    psd95_coord = peak_local_max(psd95_preprocessed, min_distance = 1, threshold_abs=psd95_threshold)

    # Rotating an image (psd95) as a control
    psd95_rot = rotate(psd95_preprocessed, 90)
    psd95_rot_coord = peak_local_max(psd95_rot, min_distance= 1, threshold_abs = psd95_threshold)
    
    # # Showing the local peaks with coordinates together with the images
    # fig, axs = plt.subplots(1, 3, figsize=(30, 30))

    # axs[0].imshow(vglut1_preprocessed, cmap='gray')
    # axs[0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    # axs[0].set_title('vglut1_pre')

    # axs[1].imshow(psd95_preprocessed, cmap='gray')
    # axs[1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    # axs[1].set_title('psd95_pre')

    # axs[2].imshow(psd95_rot, cmap='gray')
    # axs[2].plot(psd95_rot_coord[:, 1], psd95_rot_coord[:, 0], 'm.')
    # axs[2].set_title('psd95_rot')
    
    return vglut1_coord, psd95_coord, psd95_rot_coord

In [ ]:
vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(pre_1, post_1, 200, 150)
print(vglut1_coord.shape, psd95_coord.shape, psd95_rot_coord.shape)

In [ ]:

# This block of code calculates the unique colocalized VGLUT1-PSD95 count, meaning it only has unique pairs of colocalized spots.
# It checks of every VLGUT1 and PSD95 spot, whether there are any PSD95 (for VGLUT1) and VLGUT1 (for PSD95) spots in their vicinity within a specific maximum distance and counts it. 
# So it can have for one spot, multiple pairs, and thus including multi-synaptic boutons.

from scipy.spatial.distance import cdist

def count_coloc_spots(vglut1_coordinates, psd95_coordinates, pixel_size_um, max_distance_um):
    
    """Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection
    
    Args:
        vlgut1_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        psd95_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # Calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_vglut1_to_psd95 = cdist(vglut1_coordinates, psd95_coordinates)
    distances_psd95_to_vglut1 = cdist(psd95_coordinates, vglut1_coordinates)

    # Find unique colocalized spots
    colocalized_spots = set()

    # Iterate over distances from vglut1_coordinates to psd95_coordinates
    for i in range(len(vglut1_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_vglut1_to_psd95[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # Iterate over distances from psd95_coordinates to vglut1_coordinates
    for i in range(len(psd95_coordinates)):
        # Check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_psd95_to_vglut1[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # Get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count

In [ ]:
# writing a wrapper function for the two functions, which is easier to implement for the bayesian optimization
def comb_func(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold, pixel_size_um, max_distance_um):
    
    # probing the first function to get the parameters as input for the second function
    vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_threshold, psd95_threshold)
    
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(vglut1_coord, psd95_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(vglut1_coord, psd95_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_difference = abs(colocalized_spot_count - colocalized_spot_count_rot) / max(colocalized_spot_count, colocalized_spot_count_rot)
    
    return colocalized_spot_count, colocalized_spot_count_rot, scaled_difference

In [ ]:
import optuna

# setting the static variables
vglut1_preprocessed = pre_1 # Assign vglut1_pre to vglut1_preprocessed
psd95_preprocessed = post_1  # Assign psd95_pre to psd95_preprocessed
pixel_size_um = pixel_size_um  # Assign pixel_size to pixel_size_um

# Define the parameter bounds for optimization
param_bounds = {'vglut1_threshold': (500, 700), 
                'psd95_threshold': (500, 700), 
                'max_distance_um': (0.01, 1)} 

def objective(trial):
    
    vglut1_threshold = trial.suggest_float("vlgut1_threshold", 100, 200)
    psd95_threshold = trial.suggest_float("psd95_threshold", 50, 150)
    max_distance_um = trial.suggest_float("max_distance_um", 0.01, 1)
    
    _, _, scaled_spot_count_dif = comb_func(
        vglut1_preprocessed,
        psd95_preprocessed,
        vglut1_threshold,
        psd95_threshold,
        pixel_size_um,
        max_distance_um
    )

    return scaled_spot_count_dif 


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)
print(study.best_trial.value)